# Notebook Overview — Normalize and Prepare Inputs

## Purpose

This notebook normalizes the final Digital Image Processing (DIP) feature vectors and prepares classifier-ready inputs for both the **training** and **test** subsets.

---

## Inputs

The notebook expects feature-vector CSV files:

* `metadata/vectors/train_feature_vectors.csv`
* `metadata/vectors/test_feature_vectors.csv`

Paths and constants are provided by:

* `src/project_config.py`

---

## Assumptions

* Feature-vector construction has already been completed (Notebook 05)
* Each input CSV contains one row per image
* Metadata and feature columns are fully assembled
* Training and test datasets are already separated
* This notebook performs **normalization and input preparation only**
* No raw image access is required
* This notebook processes **both training and test datasets in a single run**

---

## Outputs

The following files are written under `metadata/`:

* `metadata/vectors/train_feature_vectors_normalized.csv`
* `metadata/vectors/test_feature_vectors_normalized.csv`
* `metadata/models/scaler.pkl`

These files are written to the Colab runtime and are **not automatically persisted to Google Drive**.

---

## Final Feature Structure

Each row contains:

* metadata columns:
  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* feature columns:
  * 25 normalized DIP features

---

## Notes

* Normalization is performed using **training data only**
* The same scaler is applied to both training and test data
* This ensures consistent feature scaling and prevents data leakage
* Training features are normalized to:
  * mean ≈ 0
  * standard deviation ≈ 1
* Test features are normalized using training statistics and are not expected to have mean 0 or standard deviation 1
* No classifier training is performed in this notebook
* This notebook operates entirely on structured feature data (no image access required)
* All validation steps are **fail-safe** — execution stops immediately if inconsistencies are detected

---

## Role in the Pipeline

This notebook produces **normalized feature vectors** used directly for model training and evaluation.

---

## Next Step

Proceed to:

➡️ **07_Train_Model.ipynb**

---
---

### Step 1 — Environment Setup

* Clone the GitHub repository into the local runtime (if needed)
* Import configuration from `project_config.py`
* Define input/output paths using configuration constants
* Verify required feature-vector CSV files are present

In [ ]:
# ============================================
# Step 1: Startup (Environment + Verification)
# ============================================

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Clone repo into Colab runtime (if not already present)
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_SUBSET,
    TEST_SUBSET,
    RANDOM_SEED,
    NUM_FEATURES,
    METADATA_COLUMNS,
    TRAIN_FEATURE_VECTOR_PATH,
    TEST_FEATURE_VECTOR_PATH,
    TRAIN_NORMALIZED_PATH,
    TEST_NORMALIZED_PATH,
    MODELS_METADATA_DIR,
    SCALER_PATH,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_INPUT_FILE = Path(TRAIN_FEATURE_VECTOR_PATH)
TEST_INPUT_FILE = Path(TEST_FEATURE_VECTOR_PATH)

TRAIN_OUTPUT_FILE = Path(TRAIN_NORMALIZED_PATH)
TEST_OUTPUT_FILE = Path(TEST_NORMALIZED_PATH)

MODELS_DIR = Path(MODELS_METADATA_DIR)
SCALER_FILE = Path(SCALER_PATH)

OUTPUT_DIR = TRAIN_OUTPUT_FILE.parent

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required feature-vector CSV files...\n")

required_files = [
    TRAIN_INPUT_FILE,
    TEST_INPUT_FILE,
]

missing_files = [str(file_path) for file_path in required_files if not file_path.exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing required feature-vector files:\n{missing_files}"
    )

print("All required feature-vector CSV files are present.")

if VERBOSE:
    print(f"Train input:      {TRAIN_INPUT_FILE}")
    print(f"Test input:       {TEST_INPUT_FILE}")
    print(f"Train output:     {TRAIN_OUTPUT_FILE}")
    print(f"Test output:      {TEST_OUTPUT_FILE}")
    print(f"Scaler file:      {SCALER_FILE}")
    print(f"Models directory: {MODELS_DIR}")

print("\nStartup complete.")



### Step 2 — Display Configuration

* Display input and output file paths
* Confirm runtime configuration

In [ ]:
# ============================================
# Step 2: Display Input and Output Configuration
# ============================================

print("Input files:")
print(f" - {TRAIN_INPUT_FILE}")
print(f" - {TEST_INPUT_FILE}")

print("\nOutput files:")
print(f" - {TRAIN_OUTPUT_FILE}")
print(f" - {TEST_OUTPUT_FILE}")
print(f" - {SCALER_FILE}")

print("\nConfiguration display complete.")



### Step 3 — Load Feature Vector CSVs

* Load feature-vector CSV files for:
  * training subset
  * test subset

In [ ]:
# ============================================
# Step 3: Load Train and Test Feature Vectors
# ============================================

df_train = pd.read_csv(TRAIN_INPUT_FILE)
df_test  = pd.read_csv(TEST_INPUT_FILE)

print("Train and test feature vectors loaded.\n")
print("df_train shape:", df_train.shape)
print("df_test shape: ", df_test.shape)



### Step 4 — Validate Row Counts and Structure

* Confirm consistent column structure between training and test sets
* Confirm expected feature count (25 DIP features)

In [ ]:
# ============================================
# Step 4: Validate Row Counts and Structure
# ============================================

# -------------------------------------------------
# Identify feature columns
# -------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

# -------------------------------------------------
# Validate structure consistency
# -------------------------------------------------
assert list(df_train.columns) == list(df_test.columns), \
    "Train and test column structures do not match"

print("Column structure validated.")

# -------------------------------------------------
# Validate row counts
# -------------------------------------------------
print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

# -------------------------------------------------
# Validate feature count
# -------------------------------------------------
print(f"\nNumber of metadata columns: {len(METADATA_COLUMNS)}")
print(f"Number of feature columns:  {len(feature_columns)}")

assert len(feature_columns) == NUM_FEATURES, \
    f"Expected {NUM_FEATURES} features, found {len(feature_columns)}"

print(f"\nFeature count validated ({NUM_FEATURES} DIP features).")



### Step 5 — Separate Metadata and Feature Columns

* Separate:
  * metadata columns (`filename`, `class_label`, `source_dataset`, `subset`)
  * feature columns (25 DIP features)

In [ ]:
# ============================================
# Step 5: Separate Metadata and Feature Columns
# ============================================

assert len(feature_columns) == NUM_FEATURES, \
    f"Expected {NUM_FEATURES} features, found {len(feature_columns)}"

# Separate metadata columns
train_meta = df_train[METADATA_COLUMNS].copy()
test_meta = df_test[METADATA_COLUMNS].copy()

# Separate numeric feature columns
X_train = df_train[feature_columns].copy()
X_test = df_test[feature_columns].copy()

print("Metadata and feature matrices created.\n")

print("train_meta shape:", train_meta.shape)
print("test_meta shape: ", test_meta.shape)
print("X_train shape:   ", X_train.shape)
print("X_test shape:    ", X_test.shape)

if VERBOSE:
    print("\nFirst 5 metadata rows (train):")
    display(train_meta.head())

    print("\nFirst 5 feature rows (train):")
    display(X_train.head())



### Step 6 — Fit Scaler on Training Features

* Fit a normalization scaler using **training data only**
* Compute per-feature mean and standard deviation
* Prevent data leakage by excluding test data

In [ ]:
# ============================================
# Step 6: Fit Scaler on Training Features
# ============================================

# Initialize scaler
scaler = StandardScaler()

# Fit scaler using training features only
scaler.fit(X_train)

print("Scaler fitted on training feature matrix only.")
print("This scaler will be applied to both train and test data.\n")

print("Number of features learned by scaler:", len(scaler.mean_))

if VERBOSE:
    print("\nFirst 5 feature means learned from training data:")
    for col, val in zip(feature_columns[:5], scaler.mean_[:5]):
        print(f"  {col}: {val:.6f}")

    print("\nFirst 5 feature standard deviations learned from training data:")
    for col, val in zip(feature_columns[:5], scaler.scale_[:5]):
        print(f"  {col}: {val:.6f}")



### Step 7 — Transform Feature Data

* Apply normalization to:
  * training feature matrix
  * test feature matrix
* Use the same fitted scaler for both subsets

In [ ]:
# ============================================
# Step 7: Transform Train and Test Features
# ============================================

# Apply the fitted scaler to train and test feature matrices
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames (preserve index)
df_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=feature_columns,
    index=X_train.index
)

df_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=feature_columns,
    index=X_test.index
)

print("Training and test feature matrices normalized.\n")

print("df_train_scaled shape:", df_train_scaled.shape)
print("df_test_scaled shape: ", df_test_scaled.shape)

if VERBOSE:
    print("\nFirst 5 normalized feature rows (train):")
    display(df_train_scaled.head())



### Step 8 — Recombine Metadata and Features

* Combine metadata with normalized features
* Reconstruct full DataFrames for:
  * training subset
  * test subset

In [ ]:
# ============================================
# Step 8: Recombine Metadata with Normalized Features
# ============================================

# Reset indices to ensure clean horizontal concatenation
train_meta = train_meta.reset_index(drop=True)
test_meta = test_meta.reset_index(drop=True)
df_train_scaled = df_train_scaled.reset_index(drop=True)
df_test_scaled = df_test_scaled.reset_index(drop=True)

# Recombine metadata and normalized feature columns
df_train_normalized = pd.concat([train_meta, df_train_scaled], axis=1)
df_test_normalized = pd.concat([test_meta, df_test_scaled], axis=1)

print("Metadata recombined with normalized feature columns.\n")

print("df_train_normalized shape:", df_train_normalized.shape)
print("df_test_normalized shape: ", df_test_normalized.shape)

if VERBOSE:
    print("\nFirst 5 rows of normalized training table:")
    display(df_train_normalized.head())


### Step 9 — Save Normalized Outputs

* Save:
  * `train_feature_vectors_normalized.csv`
  * `test_feature_vectors_normalized.csv`
  * `scaler.pkl`
* Verify output file creation


In [ ]:
# ============================================
# Step 9: Save Normalized Feature Vectors and Scaler
# ============================================

# -------------------------------------------------
# Save normalized feature vectors
# -------------------------------------------------
df_train_normalized.to_csv(TRAIN_OUTPUT_FILE, index=False)
df_test_normalized.to_csv(TEST_OUTPUT_FILE, index=False)

# -------------------------------------------------
# Save scaler
# -------------------------------------------------
joblib.dump(scaler, SCALER_FILE)

# -------------------------------------------------
# Verify saved files
# -------------------------------------------------
if not TRAIN_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Train normalized file not created: {TRAIN_OUTPUT_FILE}")

if not TEST_OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Test normalized file not created: {TEST_OUTPUT_FILE}")

if not SCALER_FILE.exists():
    raise FileNotFoundError(f"Scaler file not created: {SCALER_FILE}")

# -------------------------------------------------
# Confirmation output
# -------------------------------------------------
print("Saved normalized feature vectors and scaler.\n")

print(f"Train normalized CSV: {TRAIN_OUTPUT_FILE}")
print(f"Test normalized CSV:  {TEST_OUTPUT_FILE}")
print(f"Scaler saved to:      {SCALER_FILE}")



### Step 10 — Sanity Check Normalized Outputs

* Compute summary statistics for normalized features
* Verify:
  * training feature means ≈ 0
  * training feature standard deviations ≈ 1
* Confirm expected differences between training and test distributions
* Provide compact comparison of train/test statistics

In [ ]:
# ============================================
# Step 10: Sanity Check Normalized Outputs
# ============================================

# Identify normalized feature columns
normalized_feature_columns = [
    col for col in df_train_normalized.columns if col not in METADATA_COLUMNS
]

# Compute summary statistics
train_feature_means = df_train_normalized[normalized_feature_columns].mean()
train_feature_stds = df_train_normalized[normalized_feature_columns].std()

test_feature_means = df_test_normalized[normalized_feature_columns].mean()
test_feature_stds = df_test_normalized[normalized_feature_columns].std()

print("Sanity check on normalized outputs:\n")

print("Train normalized shape:", df_train_normalized.shape)
print("Test normalized shape: ", df_test_normalized.shape)

# -------------------------------------------------
# Compact normalization check
# -------------------------------------------------
mean_abs = train_feature_means.abs().mean()
std_dev = (train_feature_stds - 1).abs().mean()

print(f"\nTrain mean deviation from 0 (avg abs): {mean_abs:.6f}")
print(f"Train std deviation from 1 (avg abs):  {std_dev:.6f}")

if mean_abs > 1e-3 or std_dev > 1e-3:
    print("\nWARNING: Normalization may be off (training values not near 0/1)")

# -------------------------------------------------
# Combined comparison table
# -------------------------------------------------
summary_df = pd.DataFrame({
    "train_mean": train_feature_means,
    "train_std": train_feature_stds,
    "test_mean": test_feature_means,
    "test_std": test_feature_stds,
})

if VERBOSE:
    print("\nNormalized feature summary:")
    display(summary_df.round(6))

print("\nNormalization sanity check complete.")

